# Is it a bird? Creating a model from your own data

The basic steps we'll take are:
1. Download bird and non-bird images from Kaggle (Animals-10 dataset)
2. Fine-tune a pretrained neural network to recognise these two groups
3. Try running this model on a picture of a bird and see if it works.

## Step 0. Install required packages

In [ ]:
%pip install -Uqq fastai kagglehub
# -Uqq = --upgrade, quiet, quiet -> update all packages to the latest version and suppress output


## Step 1. Download birds data set

In [ ]:
import kagglehub
from pathlib import Path
from fastcore.all import *

# Download Animals-10 dataset from Kaggle (no login needed, auto cache)
# 586MB, first download will be slow
path = Path(kagglehub.dataset_download('alessiocorrado99/animals10')) / 'raw-img'
# Path() from pathlib is used to create a path object
path
# a path object is a more convenient way to work with file paths than using strings


In [ ]:
# list all subdirectories and count the number of images in each
for dir in sorted(path.iterdir()):
    # iterdir() returns an iterator of all the files and directories in the given path
    if dir.is_dir():
        print(f'{dir.name:15s}: {len(list(dir.iterdir()))} images')


## Step 2. Train our model

In [ ]:
from fastai.vision.all import *

# Define a function to extract labels from file names
def label_func(fname):
    return 'bird' if fname.parent.name == 'gallina' else 'non-bird'

birds = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    # define data pipeline: input: image, output: category
    get_items=get_image_files,
    # get_image_files() returns a list of all image files in the given path
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    # split the data into training and validation sets, 20% for validation, random seed for reproducibility
    get_y=label_func,
    # label_func() extracts the label from the file name
    item_tfms=Resize(192, method='squish'),
    # resize all images to 192x192 pixels, squish method preserves aspect ratio
    batch_tfms=aug_transforms(size=128, min_scale=0.75)
    # apply data augmentation to the images, resize to 128x128 pixels, minimum scale of 0.75
)

dls = birds.dataloaders(path)
# create a DataLoaders object from the DataBlock, which will handle loading the data in batches for training and validation
dls.show_batch(max_n=12)
# show a batch of 12 images from the DataLoaders object, with their corresponding labels


Now we're ready to train our model. The fastest widely used computer vision model is `resnet18`. You can train this in a few minutes, even on a CPU!

fastai comes with a helpful `fine_tune()` method which automatically uses best practices for fine tuning a pre-trained model, so we'll use that.

Due to the changes from ddgs to kagglehub, it is recommended to use GPU kernel instead of CPU kernel for training the model.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

# fine_tune(3) = 1 epoch freeze (head only) + 3 epochs unfreeze (whole model)
# epoch      = number of passes through the entire training set
# train_loss = error on training set (lower = better fit to training data)
# valid_loss = error on validation set (lower = better, true measure of model quality)
# error_rate = percentage of wrong predictions on validation set
# time       = time taken for 1 epoch


# Step 3: Use our model

In [ ]:
# Test on a bird (gallina)
bird_img = (path/'gallina').ls()[0]
bird_pred, _, bird_probs = learn.predict(bird_img)
print(f'Predict: {bird_pred}')
print(f'Probability: bird={bird_probs[0]:.3f}, non-bird={bird_probs[1]:.3f}')
bird_img.to_thumb(192)


In [ ]:
# Test on a cat (gatto)
cat_img  = (path/'gatto').ls()[0]
cat_pred, _, cat_probs = learn.predict(cat_img)
print(f'Predict: {cat_pred}')
print(f'Probability: bird={cat_probs[0]:.3f}, non-bird={cat_probs[1]:.3f}')
cat_img.to_thumb(192)


### Upload your own image:

In [ ]:
from fastai.vision.widgets import FileUpload
btn_upload = FileUpload()
btn_upload


In [ ]:
img = PILImage.create(btn_upload.data[-1])
pred, _, probs = learn.predict(img)
print(f'This is a: {pred}')
print(f'Probability: bird={probs[0]:.3f}, non-bird={probs[1]:.3f}')
img.to_thumb(256)
